<a href="https://colab.research.google.com/github/HadiyaVS/Parkinsons-Voice-Detection/blob/main/parkinson_prediction_model.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [4]:
from google.colab import files
import os, zipfile

# Upload your zipped dataset
uploaded = files.upload()  # Upload voice_dataset.zip
!unzip voice_dataset.zip -d ./  # Extracts folder "voice_dataset" in Colab

Saving voice_dataset.zip to voice_dataset (1).zip
Archive:  voice_dataset.zip
   creating: ./voice_dataset/
   creating: ./voice_dataset/healthy/
  inflating: ./voice_dataset/healthy/h1.wav  
  inflating: ./voice_dataset/healthy/h10.wav  
  inflating: ./voice_dataset/healthy/h2.wav  
  inflating: ./voice_dataset/healthy/h3.wav  
  inflating: ./voice_dataset/healthy/h4.wav  
  inflating: ./voice_dataset/healthy/h5.wav  
  inflating: ./voice_dataset/healthy/h6.wav  
  inflating: ./voice_dataset/healthy/h7.wav  
  inflating: ./voice_dataset/healthy/h8.wav  
  inflating: ./voice_dataset/healthy/h9.wav  
   creating: ./voice_dataset/simulated_pd/
  inflating: ./voice_dataset/simulated_pd/u1.wav  
  inflating: ./voice_dataset/simulated_pd/u2.wav  
  inflating: ./voice_dataset/simulated_pd/u3.wav  
  inflating: ./voice_dataset/simulated_pd/u4.wav  
  inflating: ./voice_dataset/simulated_pd/u5.wav  


In [8]:
!pip install librosa scikit-learn gradio matplotlib seaborn
import os
import numpy as np
import librosa
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.ensemble import RandomForestClassifier

# -------- Feature extraction --------
def extract_features(file_path):
    audio, sr = librosa.load(file_path, duration=3)
    mfccs = librosa.feature.mfcc(y=audio, sr=sr, n_mfcc=13)
    return np.mean(mfccs.T, axis=0), mfccs  # return mfcc for visualization

# -------- Train model if dataset exists --------
if os.path.exists("voice_dataset"):
    features = []
    labels = []

    # Healthy
    for file in os.listdir("voice_dataset/healthy"):
        f, _ = extract_features(os.path.join("voice_dataset/healthy", file))
        features.append(f)
        labels.append(0)

    # Simulated PD
    for file in os.listdir("voice_dataset/simulated_pd"):
        f, _ = extract_features(os.path.join("voice_dataset/simulated_pd", file))
        f = f + np.random.normal(0, 2.0, f.shape)  # simulate PD
        features.append(f)
        labels.append(1)

    X = np.array(features)
    y = np.array(labels)

    X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

    scaler = StandardScaler()
    X_train_scaled = scaler.fit_transform(X_train)
    X_test_scaled = scaler.transform(X_test)

    model = RandomForestClassifier(n_estimators=100, random_state=42)
    model.fit(X_train_scaled, y_train)
    print("✅ Model trained from dataset")

else:
    # Dummy model for demo
    print("⚠ Dataset not found, using dummy model")
    X_dummy = np.random.rand(10,13)
    y_dummy = np.random.randint(0,2,10)
    scaler = StandardScaler().fit(X_dummy)
    model = RandomForestClassifier(n_estimators=10)
    model.fit(scaler.transform(X_dummy), y_dummy)
    import gradio as gr
import matplotlib.pyplot as plt
import seaborn as sns

def predict_parkinsons(file_path):
    # Extract features and MFCC
    features, mfcc_matrix = extract_features(file_path)
    features_scaled = scaler.transform([features])

    # Prediction
    prediction = model.predict(features_scaled)[0]
    probability = np.max(model.predict_proba(features_scaled)) * 100

    if prediction == 1:
        label = "⚠ High Risk of Parkinson's"
    else:
        label = "✅ Low Risk (Healthy)"

    # Plot MFCC heatmap
    fig, ax = plt.subplots()
    sns.heatmap(mfcc_matrix, cmap="viridis", ax=ax)
    ax.set_title("MFCC Heatmap")

    return f"{label}\nConfidence: {probability:.2f}%", fig

# Launch Gradio
iface = gr.Interface(
    fn=predict_parkinsons,
    inputs=gr.Audio(sources="upload", type="filepath"),  # filepath avoids errors, corrected 'source' to 'sources'
    outputs=["text", "plot"],
    title="🎤 Parkinson's Voice Detection",
    description="Upload a WAV file to predict if the person is healthy or Parkinson affected."
)

iface.launch()

✅ Model trained from dataset
It looks like you are running Gradio on a hosted Jupyter notebook, which requires `share=True`. Automatically setting `share=True` (you can turn this off by setting `share=False` in `launch()` explicitly).

Colab notebook detected. To show errors in colab notebook, set debug=True in launch()
* Running on public URL: https://802771fba3835f5c2b.gradio.live

This share link expires in 1 week. For free permanent hosting and GPU upgrades, run `gradio deploy` from the terminal in the working directory to deploy to Hugging Face Spaces (https://huggingface.co/spaces)
